In [1]:
## Llama_cpp run cell

import subprocess
import atexit
import threading
import requests
import sseclient
import json

url = "http://127.0.0.1:5175/v1/chat/completions"
url2 = "http://127.0.0.1:5176/v1/chat/completions"

process = subprocess.Popen([
    "../../../QullyChat/llama/llama-server",
    "-m", "../../../QullyChat/models/Qwen3-1.7B-Q8_0.gguf",
    "-b", "512",
    "--n-gpu-layers", "0",
    "-t", "12",
    "-n", "-1",
    "--reasoning-budget", "0",
    "--port", "5175",
    "--verbose"
], stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, bufsize=1, text=True, encoding="utf-8", start_new_session=True)

process2 = subprocess.Popen([
    "../../../QullyChat/llama/llama-server",
	"-m", "../../../QullyChat/models/Qwen3-14B-Q4_K_M.gguf",
	"-b", "512",
	"--n-gpu-layers", "1000",
	"-t", "12",
	"-n", "-1",
	"--reasoning-budget", "0",
	"--port", "5176",
	"--verbose"
], stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, bufsize=1, text=True, encoding="utf-8", start_new_session=True)

atexit.register(process.terminate)
atexit.register(process2.terminate)

<bound method Popen.terminate of <Popen: returncode: None args: ['../../../QullyChat/llama/llama-server', '-m...>>

In [19]:
response = requests.post(url, json={"messages": [{'role': 'system', 'content': 'You are a professor of logic. You are teaching your students about syllogistics. You respond as if speaking to them'}, {'role': 'user', 'content': 'Give 5 examples of non-obvious complex syllogistic reasoning. Don\'t explain anything, just respond with 5 examples.'}], "max_tokens": -1})
reply = response.json()['choices'][0]['message']['content']
if "</think>" in reply:
    reply = reply.split("</think>")[1].strip()
response2 = requests.post(url2, json={"messages": [{'role': 'system', 'content': 'You are a professor of logic. You are teaching your students about syllogistics. You respond as if speaking to them'}, {'role': 'user', 'content': f'Your students gave a speech showing 5 examples of unusual syllogistic reasoning:\n{reply}\n\nCheck their correctness and provide them with the pattern of the syllogistic reasoning for those examples.'}], "max_tokens": -1})
reply2 = response2.json()['choices'][0]['message']['content']
if "</think>" in reply2:
    reply2 = reply2.split("</think>")[1].strip()

In [ ]:
print(reply)

1. All humans are mortal. All gods are mortal. Therefore, all humans are gods.  
2. All dogs are mammals. All mammals are animals. Therefore, all dogs are animals.  
3. All A are B. All C are D. Therefore, all A are D.  
4. All A are B. All B are C. Therefore, all A are C.  
5. All A are B. All C are A. Therefore, all C are B.


In [ ]:
print(reply2)

Let us examine each example carefully and analyze their logical validity and syllogistic patterns. Remember that a valid syllogism must satisfy three conditions: **three distinct terms**, **the middle term must be distributed at least once**, and **the conclusion must not contain a term that is not distributed in the premises**.

---

### **1. All humans are mortal. All gods are mortal. Therefore, all humans are gods.**  
**Validity:** ❌ **Invalid**  
**Pattern:** This is an example of the **fallacy of the undistributed middle**.  
- The premises share the same predicate (*mortal*), but the conclusion attempts to equate *humans* and *gods* (subjects).  
- The middle term (*mortal*) is **not distributed** in either premise (both are universal affirmatives, so the predicate is not distributed).  
- This is a classic logical error, often called the "fallacy of the undistributed middle."

---

### **2. All dogs are mammals. All mammals are animals. Therefore, all dogs are animals.**  
**Va

## Uwagi
Zrobiłem kilka wykonań kodu, gdy poprawiałem prompty i ten pierwszy model nigdy nie podał 5 poprawnych sylogizmów, zawsze jakieś były niepoprawne.  
Ale co ciekawe nie wynika to z błędów modelu. W pierwszej wersji promptu, model dawał przykłady sylogizmów z pełnym opisem i objaśnieniami. W tych objaśnieniach można było przeczytać, że model wie, że sylogizm jest błędny i czemu zdecydował się na podanie błędnego sylogizmu. Każdy taki przypadek wynikał z tego, iż model uznał, że błąd danego typu jest właśnie tą nieoczywistością z polecenia.  
Zastanawiałem się, czy dostosować prompt, żeby model podawał tylko poprawne sylogizmy, ale ostatecznie zdecydowałem się zostawić taki stan, ponieważ w poleceniu jest wskazane "nieoczywiste przykłady rozumowań logistycznych", a nie "poprawn nieoczywiste przykłady rozumowań logistycznych".

# Kopia odpowiedzi pierwszego modelu:
1. All humans are mortal. All gods are mortal. Therefore, all humans are gods.  
2. All dogs are mammals. All mammals are animals. Therefore, all dogs are animals.  
3. All A are B. All C are D. Therefore, all A are D.  
4. All A are B. All B are C. Therefore, all A are C.  
5. All A are B. All C are A. Therefore, all C are B.

# Kopia odpowiedzi drugiego modelu:
Let us examine each example carefully and analyze their logical validity and syllogistic patterns. Remember that a valid syllogism must satisfy three conditions: **three distinct terms**, **the middle term must be distributed at least once**, and **the conclusion must not contain a term that is not distributed in the premises**.

---

### **1. All humans are mortal. All gods are mortal. Therefore, all humans are gods.**  
**Validity:** ❌ **Invalid**  
**Pattern:** This is an example of the **fallacy of the undistributed middle**.  
- The premises share the same predicate (*mortal*), but the conclusion attempts to equate *humans* and *gods* (subjects).  
- The middle term (*mortal*) is **not distributed** in either premise (both are universal affirmatives, so the predicate is not distributed).  
- This is a classic logical error, often called the "fallacy of the undistributed middle."

---

### **2. All dogs are mammals. All mammals are animals. Therefore, all dogs are animals.**  
**Validity:** ✅ **Valid**  
**Pattern:** **Barbara (AAA-1)**  
- Premise 1: All *dogs* (A) are *mammals* (B).  
- Premise 2: All *mammals* (B) are *animals* (C).  
- Conclusion: All *dogs* (A) are *animals* (C).  
- This is a classic syllogism in the **first figure** (Barbara), where the middle term (*mammals*) is distributed in the second premise, ensuring a valid conclusion.

---

### **3. All A are B. All C are D. Therefore, all A are D.**  
**Validity:** ❌ **Invalid**  
**Pattern:** **No valid syllogism**  
- The premises are about two disjoint categories (*A-B* and *C-D*), with **no overlap** in terms.  
- The syllogism lacks a **common middle term** to connect the premises to the conclusion.  
- This is an example of the **fallacy of the illicit process** or **non sequitur** (the conclusion does not logically follow).

---

### **4. All A are B. All B are C. Therefore, all A are C.**  
**Validity:** ✅ **Valid**  
**Pattern:** **Barbara (AAA-1)**  
- Identical to example 2. This is the same **Barbara** syllogism, with the middle term (*B*) distributed in the second premise, leading to a valid conclusion.

---

### **5. All A are B. All C are A. Therefore, all C are B.**  
**Validity:** ✅ **Valid**  
**Pattern:** **Celarent (EAE-1)**  
- Premise 1: All *A* are *B* (A-B).  
- Premise 2: All *C* are *A* (C-A).  
- Conclusion: All *C* are *B* (C-B).  
- This is a valid syllogism in the **first figure**, known as **Celarent**, where the middle term (*A*) is distributed in the second premise, ensuring the conclusion follows.

---

### **Summary of Validity and Patterns**  
| Example | Valid? | Pattern/Reason |  
|--------|--------|----------------|  
| 1 | ❌ Invalid | Fallacy of the undistributed middle |  
| 2 | ✅ Valid | Barbara (AAA-1) |  
| 3 | ❌ Invalid | No common middle term (non sequitur) |  
| 4 | ✅ Valid | Barbara (AAA-1) |  
| 5 | ✅ Valid | Celarent (EAE-1) |  

---

### **Key Takeaways for Students**  
- A valid syllogism requires **three distinct terms** and a **distributed middle term**.  
- Be cautious of conclusions that equate two categories unless they are logically linked through a shared predicate.  
- The **first figure** (where the middle term is the predicate of the first premise and the subject of the second) often yields valid syllogisms (e.g., Barbara and Celarent).  

Let me know if you'd like to explore other syllogistic forms or fallacies!

In [67]:
import random

def gen_syllogism(num):
    ops = [random.choice(['a', 'i', 'e', 'o']) for x in range(num-2)]
    ops.append('a')
    ops.append('a')
    random.shuffle(ops)
    ops.append(random.choice(['a', 'i', 'e', 'o']))
    syl = ""
    for i in range(num):
        if i == 0:
            syl = f'S {ops[0]} M1'
        elif i+1 == num:
            syl += f'\nM{i} {ops[i]} P'
            syl += f'\n---------------\nS {ops[i+1]} P'
        else:
            syl += f'\nM{i} {ops[i]} M{i+1}'
        if i+1 == num:
            syl += f'\n'
    return syl

response = requests.post(url2, json={'messages': [{'role': 'system', 'content': 'You are a professor of logic and your job is to evaluate syllogisms. A - All x are y; E - No x are y; I - Some x are y; O - Some x are not y. Check if they\'re true and explain their reasoning'}, {'role': 'user', 'content': f'Check this syllogism:\n{gen_syllogism(5)}\nAnd provide a text, non-symbolic version of this syllogism, replacing all S, P and M symbolls with some random names.\n\nProvide a brief, not long of a response!'}], 'max_tokens': -1})
response.json()

{'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'role': 'assistant',
    'content': '<think>\nOkay, let\'s see. The user provided a syllogism with multiple premises and wants me to check its validity and provide a non-symbolic version.\n\nFirst, the syllogism structure is:\n\nS e M1\nM1 a M2\nM2 a M3\nM3 a M4\nM4 a P\n---------------\nS e P\n\nSo, each premise is a categorical statement. The first is "No S are M1" (E), then "All M1 are M2" (A), "All M2 are M3" (A), "All M3 are M4" (A), "All M4 are P" (A). The conclusion is "No S are P" (E).\n\nI need to check if the conclusion logically follows from the premises. Let me break it down step by step. \n\nStarting with the first premise: No S are M1. Then, since all M1 are M2, that would mean S and M1 are disjoint. But since M1 is a subset of M2, S could still be in M2. Wait, but the second premise says all M1 are M2. So if S is not in M1, but M1 is in M2, S could be in M2 or not. But the next premises take us through 

In [68]:
reply = response.json()['choices'][0]['message']['content']
if "</think>" in reply:
    reply = reply.split("</think>")[1].strip()
print(reply)

The syllogism is **invalid**. The premises do not logically entail the conclusion because the chain of "All" statements (M1→M2→M3→M4→P) does not exclude S from P, even though S is disjoint from M1. The conclusion "No S are P" is not necessarily true. 

**Non-symbolic version**:  
No *Dogs* are *Birds*.  
All *Birds* are *Animals*.  
All *Animals* are *Living Things*.  
All *Living Things* are *Organisms*.  
All *Organisms* are *Cats*.  
→ No *Dogs* are *Cats*.  

(Conclusion is logically unsupported.)
